# Hypothesis - Genre Popularity

Find which genres have the highest and lowest average popularity.

Calculate average popularity by genre
Identify highest average popularity genres
Identify lowest average popularity genres

## Inputs

CSV file used:

spotifydataset_Visualisation.csv


## Outputs

Plots for testing validity of each hypothesis before considering visualisation


## Additional Comments

Developed an experimental ETL library which is in this project (modETL_library.py) 
Has lots of cool features so will be interesting to see how it works "in the field"

In [1]:
#import libraries
import os
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import kruskal
#added by me for visualisation fine tuning
from matplotlib.ticker import MultipleLocator
from scipy.stats import f_oneway
from scipy.stats import pearsonr
from scipy.stats import spearmanr

#added by me for plotly.express visualisation issue
import nbformat 

#below solution provided by chatGPT 
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root / "assets" / "python_files") not in sys.path:
    sys.path.insert(0, str(project_root / "assets" / "python_files"))
    
import modGlobal
import modETL_Library as modETL
#end solution provided by chatGPT

# Section 1 - Initalisation

## Intitialise All Variables To Be Used In Global Stack 

In [2]:
#DataFrame variables for EDA
dfSpotify_DataSet_Work = None
dfSpotify_DataSet_Temp = None
dfSpotify_DataSet_Temp1 = None
dfSpotify_DataSet_Temp2 = None
dfSpotify_DataSet_Merged = None

#stores current directory
strCurrentDir = ""

#other vars
dictDataFrames = dict()
fig = None
axis = None


## Set Current Directory To Base Project Directory

In [3]:
#get project directory - default is jupyter notebook sub folder as that is where this file is located!
#so move back one to the project root path
# Source - https://stackoverflow.com/a/17726833
# Posted by chimpsarehungry
# Retrieved 2026-07-05, License - CC BY-SA 3.0

#get current folder
strCurrentDir = os.getcwd()

#is the last part of the path the project directory?
if not strCurrentDir.endswith(modGlobal.CNST_STR_PROJECT_DIR):
   #get current working directory and move back one to the project root path
   strCurrentDir =  os.path.normpath(os.getcwd() + os.sep + os.pardir)
   os.chdir(os.path.dirname(strCurrentDir))
   #change directory
   os.chdir(strCurrentDir)

#confirm current directory is project directory
print(f"Current Directory: \n {os.getcwd()}")

Current Directory: 
 /Users/sahraosman/Documents/vscode-projects/Spotify Music Trend Analysis


# Section 2

- Read csv file
- Look at hypothesis validity
- Use plot(s) to validate findings

## Read csv File Into Variable For Processing

In [4]:
#read csv file into DataFrame
dictDataFrames = modETL.funcReadVisualisationFilesReturnDictionary()
dfSpotify_DataSet = dictDataFrames["spotifydataset_Visualisation.csv"]

#create copy of the original DataFrame to work with
dfSpotify_DataSet_Work = dfSpotify_DataSet.copy()

2 csv Files Read Into DataFrames

DataFrames Created:
spotifydataset_Visualisation.csv
spotify_dashboard_data.csv




# Calculate Average Popularity By Genre

In [5]:
#calculate average popularity by genre

#get records wiithout duplicates
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by="popularity", ascending=False).drop_duplicates(
    subset=["artists","album_name","track_name"], keep="first")

#get mean popularity by genre
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.groupby("track_genre").agg({"popularity": "mean"}).reset_index()

#sort by popularity
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.sort_values(by="popularity", ascending=False)

#show data
dfSpotify_DataSet_Temp

,track_genre,popularity
65,k-pop,58.906250
81,pop-film,57.418972
15,chill,53.676884
94,sad,51.912258
99,singer-songwriter,50.732899
...,...,...
24,detroit-techno,11.156965
64,jazz,10.222087
67,latin,8.838565
93,romance,3.547065


# Observations

K-pop and pop-film are top with Iranian music genre at the bottom, lets put this data into a plot

In [6]:
#calculate average popularity by genre

#get records wiithout duplicates
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by="popularity", ascending=False).drop_duplicates(
    subset=["artists","album_name","track_name"], keep="first")

#get mean popularity by genre
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.groupby("track_genre").agg({"popularity": "mean"}).reset_index()

#sort by popularity
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.sort_values(by="popularity", ascending=False)

#put into a plot
fig = px.bar(dfSpotify_DataSet_Temp, x="track_genre", y="popularity", color="track_genre",
             title="Average Popularity By Genre", labels={"track_genre": "Genre", "popularity": "Average Popularity"})
fig.update_layout(xaxis_title="Genre", yaxis_title="Average Popularity", showlegend=False, height=600, width=2000)
fig.show()

# Identify Highest Average Popularity Genres

Look at top 10 most popular genres

In [7]:
#Identify highest average popularity genres
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by="popularity", ascending=False).drop_duplicates(
    subset=["artists","album_name","track_name"], keep="first")

#get mean popularity by genre
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.groupby("track_genre").agg({"popularity": "mean"}).reset_index()  

#sort by popularity
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.sort_values(by="popularity", ascending=False).head(10)  

#show data
dfSpotify_DataSet_Temp

,track_genre,popularity
65,k-pop,58.906250
81,pop-film,57.418972
15,chill,53.676884
94,sad,51.912258
99,singer-songwriter,50.732899
44,grunge,50.261029
47,hard-rock,50.218695
83,progressive-house,49.708738
55,indian,49.594828
71,metal,49.314103


# Observations

K-pop and pop-film fill the top 2 places but not by much, a wide range of genres closely pack the top 10 with on whole very little statistically between them, lets put this into a plot

In [8]:
#show plot of top 10 genres by average popularity
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by="popularity", ascending=False).drop_duplicates(
    subset=["artists","album_name","track_name"], keep="first")

#get mean popularity by genre
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.groupby("track_genre").agg({"popularity": "mean"}).reset_index()  

#sort by popularity
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.sort_values(by="popularity", ascending=False).head(10)  

#put into a plot
fig = px.bar(dfSpotify_DataSet_Temp, x="track_genre", y="popularity", color="track_genre",
             title="Top 10 Genres By Popularity", labels={"track_genre": "Genre", "popularity": "Average Popularity"})
fig.update_layout(xaxis_title="Genre", yaxis_title="Average Popularity", showlegend=False, height=600, width=1000)
fig.show()

# Observations

K-pop and pop-film rule the roost where as the other genres are quite tightly packed together statistically speaking


# Identify Lowest Average Genres By Popularity

Look at the bottom 10 genres by popularity

In [9]:
#Identify lowest average popularity genres
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by="popularity", ascending=False).drop_duplicates(
    subset=["artists","album_name","track_name"], keep="first")

#get mean popularity by genre
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.groupby("track_genre").agg({"popularity": "mean"}).reset_index()  

#sort by popularity
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.sort_values(by="popularity", ascending=True).head(10)  

#show data
dfSpotify_DataSet_Temp

,track_genre,popularity
59,iranian,2.247444
93,romance,3.547065
67,latin,8.838565
64,jazz,10.222087
24,detroit-techno,11.156965
13,chicago-house,12.144764
16,classical,12.215844
42,grindcore,14.576613
66,kids,14.905858
54,idm,15.676142


Observations

Iranian and romance are really low in popularity, could be worth asking questions as to if these genres are limited to a small number of artist or not, lets put this into a plot

In [11]:
#Identify lowest average popularity genres
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by="popularity", ascending=False).drop_duplicates(
    subset=["artists","album_name","track_name"], keep="first")

#get mean popularity by genre
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.groupby("track_genre").agg({"popularity": "mean"}).reset_index()  

#sort by popularity
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.sort_values(by="popularity", ascending=True).head(10)  

#put into a plot
fig = px.bar(dfSpotify_DataSet_Temp, x="track_genre", y="popularity", color="track_genre",
             title="Bottom 10 Genres By Popularity", labels={"track_genre": "Genre", "popularity": "Average Popularity"})
fig.update_layout(xaxis_title="Genre", yaxis_title="Average Popularity", showlegend=False, height=600, width=1000)
fig.show()

# Observations

We can clealry see just how down the popularity order Iranian and Romance genres are in the list, is there are feasible business reason to keep music in these genres on the servers?